# DBT Testing

In [5]:
from collections.abc import Callable, Sequence
from typing import Optional
import dagster as dg

# 1. Paste the original code here
@dg.template_var
def group_from_fqn() -> Callable[[Sequence[str]], str | None]:
    """Returns a function that extracts the group name from a dbt model's fqn (fully qualified name).

    The fqn contains the directory structure, e.g.:
    ["jaffle_shop", "staging", "stg_customers"] -> returns "staging"
    ["jaffle_shop", "marts", "customers"] -> returns "marts"
    """

    def _get_group(fqn: Sequence[str]) -> str | None:
        # fqn structure: [project_name, folder, ..., model_name]
        # We want the first folder after the project name
        if len(fqn) >= 2:
            return fqn[1]  # Returns "staging", "intermediate", "marts", etc.
        return None

    return _get_group


# 2. Call the decorated function to get the inner _get_group function
get_group = group_from_fqn()

# 3. Test it with sample fqn lists matching your project structure
print(get_group(["domain_data_products", "comics", "staging"]))          # Expected: comics
print(get_group(["domain_data_products", "jaffle_shop", "staging"]))   # Expected: jaffle_shop
print(get_group(["domain_data_products"]))                                   # Expected: None

comics
jaffle_shop
None


In [7]:
from jinja2 import Template
from collections.abc import Sequence

# 1. Bring in your custom group extraction function
def group_from_fqn():
    def _get_group(fqn: Sequence[str]) -> str | None:
        if len(fqn) >= 2:
            return fqn[1]
        return None
    return _get_group

# 2. Set up a mock Jinja environment / context matching your real structure
context_data = {
    "group_from_fqn": group_from_fqn,
    "node": {
        "name": "int_comics",
        "fqn": ["domain_data_products", "comics", "intermediate", "int_comics"]
    }
}

# 3. Define your template strings from def.yaml
group_template = Template("{{ group_from_fqn()(node.fqn) }}")
desc_template = Template("Transforms data using dbt model {{ node.name }}")

# 4. Render them to see the output
print("Evaluated Group Name:", group_template.render(context_data))
print("Evaluated Description:", desc_template.render(context_data))

Evaluated Group Name: comics
Evaluated Description: Transforms data using dbt model int_comics


In [12]:
from pathlib import Path
import dagster as dg
from xkcd_dagster.defs.transformation_dbt.template_vars import group_from_fqn

# 1. Target direct dbt location
MODELS_DIR = Path("/home/maxantipev/analytics-engineering/transformation_dbt/domain_data_products/models")

# 2. Instantiate extractor
extractor = group_from_fqn()

# 3. Extract groups dynamically from subfolders
all_groups = [
    extractor([MODELS_DIR.parent.name, folder.name])
    for folder in MODELS_DIR.iterdir()
    if folder.is_dir()
]

print("Discovered Groups:", all_groups)

# 4. Simulate job creation
jobs = {}
for group in all_groups:
    jobs[f"{group}_job"] = dg.define_asset_job(
        name=f"{group}_job",
        selection=dg.AssetSelection.groups(group),
    )

print("Generated Jobs:", list(jobs.keys()))

Discovered Groups: ['comics', 'jaffle_shop']
Generated Jobs: ['comics_job', 'jaffle_shop_job']


In [9]:
import dagster as dg
import xkcd_dagster.defs as defs_module

# 1. Load the project definitions from your defs module
defs = dg.load_definitions_from_module(defs_module)

# 2. Extract unique group names across all assets loaded by def.yaml / template_vars.py
all_groups = {
    spec.group_name
    for asset in defs.assets
    for spec in (asset.specs if hasattr(asset, "specs") else [])
    if spec.group_name
}

print("Discovered Groups:", all_groups)

# 3. Simulate job creation for each discovered group
jobs = {}
for group in all_groups:
    jobs[f"{group}_job"] = dg.define_asset_job(
        name=f"{group}_job",
        selection=dg.AssetSelection.groups(group),
    )

print("Generated Jobs:", list(jobs.keys()))

Discovered Groups: set()
Generated Jobs: []


/tmp/ipykernel_170728/464146548.py:5: PreviewWarning: Function `load_definitions_from_module` is currently in preview, and may have breaking changes in patch version releases. This feature is not considered ready for production use.
  defs = dg.load_definitions_from_module(defs_module)
/home/maxantipev/analytics-engineering/.venv/lib/python3.10/site-packages/dagster/_core/definitions/module_loaders/load_defs_from_module.py:77: PreviewWarning: Function `load_definitions_from_modules` is currently in preview, and may have breaking changes in patch version releases. This feature is not considered ready for production use.
  return load_definitions_from_modules(
